# Water Segmentation Using Optical and Multispectral Satellite Data
## Using Transfer Learning

This notebook demonstrates how to **segment water bodies** from satellite imagery using **multispectral and optical data**. 

Accurate water segmentation is important for:
- Flood monitoring and management  
- Environmental conservation  
- Water resource planning  

I used a Harmonized Sentinel-2/Landsat dataset where each sample includes:

- 12 channels (spectral bands + geospatial layers)
    - Coastal aerosol
    - Blue
    - Green
    - Red
    - NIR
    - SWIR1
    - SWIR2
    - QA BAND
    - Merit DEM
    - Copernicus DEM
    - ESA world cover map
    - Water occurrence probebility
- A binary mask indicating ,water (1) / non-water (0)

Ground sampling distance = 30 m
Patch size = 512 pixels



In [36]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import os
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
import torchvision.transforms.functional as TF
import rasterio


In [37]:
import segmentation_models_pytorch as smp
print(smp.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

0.5.0
Using device: cpu


In [38]:
image_dir = r"C:\Users\walaa\Desktop\Cellula CV\Water Segmentation\Transfer Learning\data\images"
label_dir = r"C:\Users\walaa\Desktop\Cellula CV\Water Segmentation\Transfer Learning\data\labels"

image_files = sorted(os.listdir(image_dir))
mask_files = sorted(os.listdir(label_dir))
print(f"{len(image_files)} images and {len(mask_files)} masks")

306 images and 456 masks


In [39]:
images_list = []
masks_list = []

for img_file, mask_file in tqdm(zip(image_files, mask_files), total=len(image_files)):
    with rasterio.open(os.path.join(image_dir, img_file)) as src:
        img_arr = src.read().astype(np.float32)  # shape: (bands, H, W)

    resized_bands = []
    for band in img_arr:
        resized_bands.append(cv2.resize(band, (128, 128), interpolation=cv2.INTER_LINEAR))
    img_arr = np.stack(resized_bands)
    images_list.append(img_arr)

    mask_arr = np.array(Image.open(os.path.join(label_dir, mask_file))).astype(np.float32)
    mask_arr = cv2.resize(mask_arr, (128, 128), interpolation=cv2.INTER_NEAREST)
    masks_list.append(mask_arr)

#convert to numpy arrays
X = np.array(images_list)         # (N, 12, 128, 128)
y = np.array(masks_list)[:, np.newaxis, :, :]  # (N, 1, 128, 128)

print("Images shape:", X.shape, "Masks shape:", y.shape)


100%|██████████| 306/306 [00:05<00:00, 57.64it/s]


Images shape: (306, 12, 128, 128) Masks shape: (306, 1, 128, 128)


### Normalisation

In [40]:
means = X.mean(axis=(0,2,3))
stds  = X.std(axis=(0,2,3))
for i in range(X.shape[1]):
    X[:, i, :, :] = (X[:, i, :, :] - means[i]) / (stds[i] + 1e-8)

print("Normalization done.")

Normalization done.


### Train, Val, Test split

In [41]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (214, 12, 128, 128) Val: (46, 12, 128, 128) Test: (46, 12, 128, 128)


### Augmentation and Shuffle

In [42]:
def augment(img, mask):
    img = torch.tensor(img, dtype=torch.float32)
    mask = torch.tensor(mask, dtype=torch.float32)

    if np.random.rand() > 0.5:
        img = TF.hflip(img)
        mask = TF.hflip(mask)
    if np.random.rand() > 0.5:
        img = TF.vflip(img)
        mask = TF.vflip(mask)
    angle = np.random.uniform(-20, 20)
    img = TF.rotate(img, angle)
    mask = TF.rotate(mask, angle)
    return img, mask

X_train_aug_list, y_train_aug_list = [], []

for i in range(len(X_train)):
    img, mask = augment(X_train[i], y_train[i])
    X_train_aug_list.append(img.numpy())
    y_train_aug_list.append(mask.numpy())

X_train_combined = np.concatenate([X_train, np.array(X_train_aug_list)], axis=0)
y_train_combined = np.concatenate([y_train, np.array(y_train_aug_list)], axis=0)



indices = np.arange(len(X_train_combined))
np.random.shuffle(indices)
X_train_combined = X_train_combined[indices]
y_train_combined = y_train_combined[indices]


### DataLoaders

In [43]:
train_dataset = TensorDataset(torch.tensor(X_train_combined, dtype=torch.float32),
                              torch.tensor(y_train_combined, dtype=torch.float32))
val_dataset   = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                              torch.tensor(y_val, dtype=torch.float32))
test_dataset  = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                              torch.tensor(y_test, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)

### Model, loss and optimizer

In [44]:
model = smp.DeepLabV3Plus(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=X.shape[1],
    classes=1
)

# model.segmentation_head = nn.Sequential(
#     nn.Dropout(0.5),
#     model.segmentation_head
# )

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

for param in model.parameters():
    param.requires_grad = True

### Training

In [ ]:
num_epochs = 10
best_val_loss = float("inf")

for epoch in range(num_epochs):
    #train
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Train"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = torch.sigmoid(outputs) > 0.5
        correct += (preds == masks).sum().item()
        total += masks.numel()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # validate
    model.eval()
    val_running_loss = 0.0
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Val"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_running_loss += loss.item()
            preds = torch.sigmoid(outputs) > 0.5
            val_correct += (preds == masks).sum().item()
            val_total += masks.numel()

    val_loss = val_running_loss / len(val_loader)
    val_acc = val_correct / val_total

    #save
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "deeplabv3plus_best.pth")

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

Train Loss: 0.3681, Train Acc: 0.8388 | Val Loss: 0.9683, Val Acc: 0.5889

In [45]:
model.load_state_dict(torch.load("C:\\Users\\walaa\\Desktop\\Cellula CV\\Water Segmentation\\Transfer Learning\\Model\\deeplabv3plus_best.pth", map_location=device))

<All keys matched successfully>

In [46]:
model.eval()

test_loss = 0.0
correct, total = 0, 0
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        loss = criterion(outputs, masks)
        test_loss += loss.item()
        preds = torch.sigmoid(outputs) > 0.5
        correct += (preds == masks).sum().item()
        total += masks.numel()

test_loss /= len(test_loader)
test_acc = correct / total
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Test Loss: 0.6451, Test Accuracy: 0.6265
